<a href="https://colab.research.google.com/github/shmuhammadd/semantic_relatedness/blob/main/Simple_English_Baseline_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple Co-Occurance Baseline for Semantic Relatedness -- English Example

Authors: Krishnapriya Vishnubhotla, Mohamed Abdalla

Introduction:

In this starter notebook, we will take you through the process of estimating semantic relatedness using simple co-occurance baselines. The notebook was adapted from a notebook for SemEval 2023 Shared Task 12: AfriSenti (Task A).

### Package Imports

In [1]:
import re
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, pearsonr
import matplotlib.pyplot as plt
plt.style.use('ggplot')

### Data Import

The training data will have a real-values semantic textual relatedness score (between 0 and 1) for a pair of English-language sentences.

The data is structured as a CSV file with the following fields:
- PairID: a unique identifier for the sentence pair
- Text: two sentences separated by a newline ('\n') character
- Score: the semantic textual relatedness score for the two sentences

Below we will show you how to load and re-format the provided data file.

In [3]:
# Load the File
df_str_rel = pd.read_csv('Track A/dich/str_dataset.csv', usecols=[0,1,2,3])
df_str_rel.head()

,ID,sentence1,sentence2,score
0,1,Tôi đã rất tự_hào về đất_nước tôi .,Tôi đang học cách nấu một món ăn mới cho bữa tối.,0
1,2,Tôi đã rất tự_hào về đất_nước tôi .,Đất nước tôi có nhiều danh lam thắng cảnh và n...,1
2,3,Tôi đã rất tự_hào về đất_nước tôi .,Tôi cảm thấy vô cùng tự hào về quê hương của m...,2
3,4,"Tôi đã vô_cùng sợ_hãi , và có cảm_giác như tim...",Tôi đang lên kế hoạch đi du lịch cùng gia đình...,0
4,5,"Tôi đã vô_cùng sợ_hãi , và có cảm_giác như tim...","Sau sự việc đó, tôi bị mất ngủ và thường xuyên...",1


In [5]:
df_str_rel['ID']
df_str_rel['sentence1']
df_str_rel['sentence2']
df_str_rel['score']


0     0
1     1
2     2
3     0
4     1
5     2
6     0
7     1
8     2
9     0
10    1
11    2
12    0
13    1
14    2
15    0
16    1
17    2
18    0
19    1
20    2
21    0
22    1
23    2
24    0
25    1
26    2
27    0
28    1
29    2
30    0
31    1
32    2
33    0
34    1
35    2
36    0
37    1
38    2
39    0
40    1
41    2
42    0
43    1
44    2
45    0
46    1
47    2
48    0
49    1
50    2
51    0
52    1
53    2
54    0
55    1
56    2
57    0
58    1
59    2
Name: score, dtype: int64

In [6]:
df_str_rel['Split_Text'] = df_str_rel.apply(
    lambda row: [
        str(row['sentence1']),
        str(row['sentence2'])
    ],
    axis=1
)


# Dice Score (Overlap Score)

A simple baseline for estimating semantic relatedness between two sentences is to look at the proportion of words that they share in common.

There are many ways to change the score below. Consider:
1. Removing stop words and/or puncutation
2. Counting duplicate words (currently not counted)
3. Weighting rarer words differently
4. Splitting tokens differently

In [11]:
def dice_score(s1,s2):
  s1 = s1.lower()
  s1_split = re.findall(r"\w+|[^\w\s]", s1, re.UNICODE)

  s2 = s2.lower()
  s2_split = re.findall(r"\w+|[^\w\s]", s2, re.UNICODE)

  dice_coef = 2* len(set(s1_split).intersection(set(s2_split))) / (len(set(s1_split)) + len(set(s2_split)))
  return round(dice_coef, 2)

## Calculate Dice Score

In [12]:
true_scores = df_str_rel['score'].values
pred_scores = []

for index, row in df_str_rel.iterrows():
    s1 = row["sentence1"]
    s2 = row["sentence2"]

    # Overlap score
    pred_scores.append(dice_score(s1, s2))


In [13]:
# How well does the baseline correlate with human judgments?
print("Pearson Correlation:", round(pearsonr(true_scores,pred_scores)[0],2))

Pearson Correlation: 0.34


# Generate submission file

### Append prediction to dataframe

In [14]:
df_str_rel['Pred_Score'] = pred_scores
df_str_rel.head()

,ID,sentence1,sentence2,score,Split_Text,Pred_Score
0,1,Tôi đã rất tự_hào về đất_nước tôi .,Tôi đang học cách nấu một món ăn mới cho bữa tối.,0,"[Tôi đã rất tự_hào về đất_nước tôi ., Tôi đang...",0.20
1,2,Tôi đã rất tự_hào về đất_nước tôi .,Đất nước tôi có nhiều danh lam thắng cảnh và n...,1,"[Tôi đã rất tự_hào về đất_nước tôi ., Đất nước...",0.17
2,3,Tôi đã rất tự_hào về đất_nước tôi .,Tôi cảm thấy vô cùng tự hào về quê hương của m...,2,"[Tôi đã rất tự_hào về đất_nước tôi ., Tôi cảm ...",0.30
3,4,"Tôi đã vô_cùng sợ_hãi , và có cảm_giác như tim...",Tôi đang lên kế hoạch đi du lịch cùng gia đình...,0,"[Tôi đã vô_cùng sợ_hãi , và có cảm_giác như ti...",0.13
4,5,"Tôi đã vô_cùng sợ_hãi , và có cảm_giác như tim...","Sau sự việc đó, tôi bị mất ngủ và thường xuyên...",1,"[Tôi đã vô_cùng sợ_hãi , và có cảm_giác như ti...",0.25


### Generate submission file

Submission file has two columns: '**PairID**' and '**Pred_Score**'

In [15]:
df_str_rel[['ID', 'Pred_Score']].to_csv('pred_eng.csv', index=False)